In [14]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
import imutils
from imutils import paths
from imutils.video import VideoStream
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import load_img,img_to_array,to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout,Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import classification_report,confusion_matrix
import seaborn as sns

os.chdir(r"D:\proj-python\Face Mask Dataset")



In [10]:
def detect_and_predict_mask(frame, faceNet, maskNet):

    (h, w) = frame.shape[:2]

    # Caffe face detector requires 300x300
    blob = cv2.dnn.blobFromImage(
        frame,
        1.0,
        (300, 300),
        (104, 177, 123),
        swapRB=False,
        crop=False
    )

    faceNet.setInput(blob)

    detections = faceNet.forward()

    faces = []
    locs = []
    preds = []

    for i in range(0, detections.shape[2]):

        confidence = detections[0, 0, i, 2]

        if confidence < 0.5:
            continue

        box = detections[0, 0, i, 3:7] * np.array(
            [w, h, w, h]
        )

        (startX, startY, endX, endY) = box.astype("int")

        startX = max(0, startX)
        startY = max(0, startY)

        endX = min(w - 1, endX)
        endY = min(h - 1, endY)

        face = frame[
            startY:endY,
            startX:endX
        ]

        if face.size == 0:
            continue

        # BGR -> RGB
        face = cv2.cvtColor(
            face,
            cv2.COLOR_BGR2RGB
        )

        # MobileNetV2 input
        face = cv2.resize(
            face,
            (224, 224)
        )

        face = face.astype("float32") / 255.0

        faces.append(face)

        locs.append(
            (startX, startY, endX, endY)
        )

    if len(faces) > 0:

        faces = np.array(faces)

        preds = maskNet.predict(
            faces,
            verbose=0
        )

    return (locs, preds)

In [6]:

prototxtPath="deploy.prototxt.txt"
caffeweights="res10_300x300_ssd_iter_140000.caffemodel"
faceNet=cv2.dnn.readNetFromCaffe(prototxtPath,caffeweights)

print("[INFO] loading mask detector model...")
model=tf.keras.models.load_model("mask_detector.keras")
print("[INFO] model loaded successfully.")

model.summary()



[INFO] loading mask detector model...


[INFO] model loaded successfully.
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                     

In [7]:
print("FaceNet:")
print(faceNet)

FaceNet:
< cv2.dnn.Net 000001A439DF7CD0>


In [16]:
print("[INFO] starting Video stream ...")
vs=VideoStream(src=0).start()
time.sleep(2.0)

fps_start=time.time()
fps_counter=0
fps=0

while True:
    frame=vs.read()

    if frame is None:
        print("[WARNING] Cannot read frame")
        break

    frame=imutils.resize(frame,width=600)

    (locs,preds)=detect_and_predict_mask(frame,faceNet,model)

    for (box,pred) in zip(locs,preds):
        (startX,startY,endX,endY)=box
        (mask,withoutMask)=pred

        label="Mask" if mask>withoutMask else "No Mask"
        confidence=max(mask,withoutMask)
        color=(0,255,0) if label=="Mask" else (0,0,255)

        label="{}:{:.2f}%".format(label,confidence*100)

        cv2.rectangle(frame,(startX,startY),(endX,endY),color,2)
        cv2.putText(frame,label,(startX,startY-10),cv2.FONT_HERSHEY_SIMPLEX,0.45,color,2)

    fps_counter+=1
    elapsed=time.time()-fps_start

    if elapsed>=1:
        fps=fps_counter/elapsed
        fps_counter=0
        fps_start=time.time()

    cv2.putText(frame,"FPS:{:.1f}".format(fps),(10,30),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),2)
    cv2.putText(frame,"Faces:{}".format(len(locs)),(10,60),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),2)

    cv2.imshow("Mask Detection",frame)

    key=cv2.waitKey(1) & 0xFF

    if key==ord("q"):
        break

cv2.destroyAllWindows()
vs.stop()
print("[INFO] Video stream stopped.")

[INFO] starting Video stream ...
[INFO] Video stream stopped.
